# CardioBreath — Google Colab / Kaggle T4 GPU Training Notebook

Trains all **5 subject-disjoint cross-validation folds** for:
1. **Baseline 1:** `HRV + LightGBM`
2. **Baseline 2:** `Raw 1-min ECG 1D-CNN`
3. **Ablations:** `RRI-Only`, `EDR-Only`, `Dual-Stream 1-min`
4. **Proposed Model:** `CardioBreathNet` (Dual-Stream Multi-Scale 1D-CNN + BiGRU + Temperature Scaling)

Then exports the 5-fold ensemble to **ONNX**, runs the **locked held-out evaluation (`x01–x35`)**, and zips all trained artifacts for download back into your local repo.

> **Runtime:** Select `Runtime -> Change runtime type -> T4 GPU`. Total wall-clock time is ~15–20 minutes.

In [12]:
import json
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [7]:
%cd ..

d:\Tekkadan\CardioBreath_AI_for_medicine_hackathon


In [3]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

Tue Nov 24 13:48:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 617.42                 KMD Version: 617.42        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   69C    P8              6W /   64W |     345MiB /   4096MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 1. Clone CardioBreath repository (phase_2 branch)
REPO_URL = 'https://github.com/Hackeddog/CardioBreath_AI_for_medicine_hackathon.git'
BRANCH = 'phase_2'

!rm -rf CardioBreath_AI_for_medicine_hackathon
!git clone --branch {BRANCH} {REPO_URL}
%cd CardioBreath_AI_for_medicine_hackathon
!pip install -q -r requirements.txt

In [5]:
# 2. Download PhysioNet Apnea-ECG (70 nights + 8 resp-belt nights + Challenge 2000 test labels)
!python ../scripts/download_data.py --workers 8

# 3. Run biomedical DSP preprocessing + physical EDR validation
!python ../scripts/preprocess.py

[download] All 191 Apnea-ECG files already cached in D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\data\raw\apnea-ecg.
[download] Held-out test labels already present: D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\data\raw\apnea-ecg\event-2-answers.txt

=== Dataset Integrity Summary ===
Learning ECGs found : 35/35
Learning .apn found : 35/35
Held-out ECGs found : 35/35
Held-out labels     : 35/35 records (17,268 minutes)
Resp-belt records   : 8/8
[OK] PhysioNet Apnea-ECG dataset and held-out labels verified successfully.
[preprocess] Processing 35 learning recordings ...
[preprocess] Processing 35 LOCKED held-out test recordings ...
[preprocess] Running physical EDR validation against 8 respiratory-belt recordings ...
[OK] EDR Belt Validation (8 records): Mean Peak Resp Coherence = 0.251, Breathing Rate MAE = 0.94 bpm
[OK] Preprocessing complete. Cached tensors written to D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\data\processed and demo bundle to D:\Tekkadan\CardioBr


Train DSP: 100%|██████████| 35/35 [01:51<00:00,  3.20s/it]

Test DSP : 100%|██████████| 35/35 [02:01<00:00,  3.48s/it]


In [9]:
# 4. Train all baselines, ablations, and CardioBreathNet across all 5 subject-disjoint folds
!python train.py --model all --device cuda --seed 42


=== Training Baseline 1: `lgbm` (5-fold subject-disjoint CV, seed=42) ===
  [lgbm] fold_0: val_auroc=0.8810 (best_iter=89)
  [lgbm] fold_1: val_auroc=0.5790 (best_iter=16)
  [lgbm] fold_2: val_auroc=0.8793 (best_iter=300)
  [lgbm] fold_3: val_auroc=0.9601 (best_iter=383)
  [lgbm] fold_4: val_auroc=0.9059 (best_iter=209)
[DONE] lgbm: OOF AUROC=0.8665, Temp=1.206, Thr=0.350

=== Training `cardiobreath` (mode=dual_5min, device=cuda, epochs=35, folds=[0, 1, 2, 3, 4], smoke=False) ===
  [cardiobreath fold_0] ep 01/35 loss=0.3445 val_auroc=0.9494 (best=0.9494)
  [cardiobreath fold_0] ep 05/35 loss=0.1711 val_auroc=0.9399 (best=0.9494)
  [cardiobreath fold_0] Early stopping at epoch 8.
  [OK] Saved fold_0.pt | best_val_auroc=0.9494 | elapsed=53.8s
  [cardiobreath fold_1] ep 01/35 loss=0.3765 val_auroc=0.9167 (best=0.9167)
  [cardiobreath fold_1] ep 03/35 loss=0.2206 val_auroc=0.9571 (best=0.9571)
  [cardiobreath fold_1] ep 05/35 loss=0.1796 val_auroc=0.9487 (best=0.9571)
  [cardiobreath fold

c:\Users\loren\anaconda3\envs\cardiobreath\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\loren\anaconda3\envs\cardiobreath\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\loren\anaconda3\envs\cardiobreath\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\loren\anaconda3\envs\cardiobreath\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_s

In [10]:
# 5. Export CardioBreath 5-fold ensemble to ONNX (+ INT8) and benchmark CPU latency
!python scripts/export_onnx.py --model cardiobreath --quantize

  [onnx] cardiobreath_fold_0.onnx: size=1.67 MB | parity_max_diff=2.38e-07
  [onnx] cardiobreath_fold_1.onnx: size=1.67 MB | parity_max_diff=4.77e-07
  [onnx] cardiobreath_fold_2.onnx: size=1.67 MB | parity_max_diff=1.43e-06
  [onnx] cardiobreath_fold_3.onnx: size=1.67 MB | parity_max_diff=4.77e-07
  [onnx] cardiobreath_fold_4.onnx: size=1.67 MB | parity_max_diff=2.38e-07
[OK] Exported 5 ONNX models to D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\artifacts\onnx\cardiobreath | 500-min CPU latency: 854.9 ms/fold (4274.7 ms full ensemble)


c:\Users\loren\anaconda3\envs\cardiobreath\Lib\site-packages\torch\onnx\symbolic_opset9.py:4279: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  warnings.warn(


In [11]:
# 6. Evaluate 5-fold Cross-Validation first, then run SINGLE locked evaluation on x01-x35
!python evaluate.py --split val --seed 42 --device cuda
!python evaluate.py --split test --seed 42 --device cuda

print(open('reports/benchmark_table_test.md').read())

[VAL] Baseline 1: HRV + LightGBM         | AUROC=0.8712 | Acc=0.7791 | F1=0.7384 | eAHI MAE=10.16 (r=0.773) | ECE=0.0253
[VAL] Baseline 2: Raw 1-min ECG CNN      | AUROC=0.8110 | Acc=0.7314 | F1=0.6999 | eAHI MAE=12.65 (r=0.686) | ECE=0.0494
[VAL] Ablation A: RRI-Only (5-min)       | AUROC=0.9486 | Acc=0.8887 | F1=0.8590 | eAHI MAE=4.65 (r=0.950) | ECE=0.0234
[VAL] Ablation B: EDR-Only (5-min)       | AUROC=0.8608 | Acc=0.7957 | F1=0.7382 | eAHI MAE=9.40 (r=0.774) | ECE=0.0880
[VAL] Ablation C: Dual-Stream (1-min)    | AUROC=0.9106 | Acc=0.8335 | F1=0.7925 | eAHI MAE=6.11 (r=0.918) | ECE=0.0325
[VAL] CardioBreath (Dual-Stream 5-Fold)  | AUROC=0.9633 | Acc=0.9045 | F1=0.8749 | eAHI MAE=3.48 (r=0.963) | ECE=0.0340

[OK] Saved evaluation summary & figures into D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\reports
[TEST] Baseline 1: HRV + LightGBM         | AUROC=0.9088 | Acc=0.7898 | F1=0.7697 | eAHI MAE=10.33 (r=0.863) | ECE=0.0894
[TEST] Baseline 2: Raw 1-min ECG CNN      | AUROC=0

In [ ]:
# 7. Package trained checkpoints, ONNX weights, and evaluation figures for download
!zip -r cardiobreath_trained_artifacts.zip artifacts/onnx artifacts/checkpoints/lgbm artifacts/checkpoints/cardiobreath reports
from google.colab import files
files.download('cardiobreath_trained_artifacts.zip')

In [13]:
reports_dir = Path("reports")
out_pdf = reports_dir / "cardiobreath_results.pdf"

LABELS = {
    "lgbm": "Baseline 1: HRV + LightGBM",
    "raw_cnn": "Baseline 2: Raw 1-min ECG CNN",
    "ablation_rri": "Ablation A: RRI-Only (5-min)",
    "ablation_edr": "Ablation B: EDR-Only (5-min)",
    "ablation_1min": "Ablation C: Dual-Stream (1-min)",
    "cardiobreath": "CardioBreath (Dual-Stream 5-Fold)",
}
COLS = ["Model", "AUROC", "AUPRC", "Acc", "Sens", "Spec", "F1", "eAHI MAE", "r", "Scr>=5", "ECE"]

def make_rows(data):
    rows = []
    for k, name in LABELS.items():
        if k not in data["models"]:
            continue
        m = data["models"][k]["minute_metrics"]
        r = data["models"][k]["recording_metrics"]
        rows.append([
            name, f"{m['auroc']:.3f}", f"{m['auprc']:.3f}", f"{m['accuracy']:.3f}",
            f"{m['sensitivity']:.3f}", f"{m['specificity']:.3f}", f"{m['f1']:.3f}",
            f"{r['eahi_mae']:.2f}", f"{r['eahi_pearson_r']:.3f}",
            f"{r['screening_cutoff_5']['sensitivity']:.3f}", f"{m['ece']:.3f}",
        ])
    return rows

def draw_table(ax, title, cols, rows, fontsize=8, highlight_last=False):
    ax.axis("off")
    ax.set_title(title, fontsize=11, fontweight="bold", loc="left", pad=8)
    tbl = ax.table(cellText=rows, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)
    tbl.scale(1.0, 1.3)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#0F2942")
            cell.set_text_props(color="white", fontweight="bold")
        elif highlight_last and r == len(rows):
            cell.set_facecolor("#CCFBF1")
            cell.set_text_props(fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#F8FAFC")

test_data = json.loads((reports_dir / "metrics_summary_test.json").read_text(encoding="utf-8"))
val_data = json.loads((reports_dir / "metrics_summary_val.json").read_text(encoding="utf-8"))

with PdfPages(out_pdf) as pdf:
    fig = plt.figure(figsize=(11, 8.5))
    fig.suptitle("CardioBreath — Evaluation Summary", fontsize=14, fontweight="bold", y=0.97)
    draw_table(fig.add_axes([0.04, 0.52, 0.92, 0.36]), "Held-Out Test (x01–x35)", COLS, make_rows(test_data),
    highlight_last=True)
    draw_table(fig.add_axes([0.04, 0.10, 0.92, 0.36]), "5-Fold CV (a01–c10)", COLS, make_rows(val_data),
    highlight_last=True)
    pdf.savefig(fig)
    plt.close(fig)

    rec_cols = ["Record", "Min", "Usable%", "True eAHI", "Pred eAHI", "Abs Err", "True Tier", "Pred Tier", "Acc"]
    rec_rows = [
        [r["record_id"], str(r["n_minutes"]), f"{r['usable_ratio']*100:.1f}%",
            f"{r['true_eahi']:.2f}", f"{r['pred_eahi']:.2f}", f"{r['abs_err_eahi']:.2f}",
            r["true_tier"], r["pred_tier"], f"{r['minute_acc']:.3f}"]
        for r in test_data["models"]["cardiobreath"]["recording_metrics"]["per_record"]
    ]
    fig = plt.figure(figsize=(8.5, 11))
    draw_table(fig.add_axes([0.05, 0.04, 0.90, 0.90]), "CardioBreath Per-Night Test Results (x01–x35)", rec_cols,
    rec_rows, fontsize=7.5)
    pdf.savefig(fig)
    plt.close(fig)

    for fname in [
        "roc_pr_curves_test.png", "bland_altman_eahi_test.png",
        "calibration_reliability_test.png", "noise_robustness_sqi_test.png",
        "edr_belt_validation.png", "failure_cases_test.png",
    ]:
        fpath = reports_dir / "figures" / fname
        if fpath.exists():
            fig, ax = plt.subplots(figsize=(11, 8.5))
            ax.imshow(mpimg.imread(fpath))
            ax.axis("off")
            ax.set_title(fname, fontsize=11, fontweight="bold")
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

print(f"Saved: {out_pdf.resolve()}")

Saved: D:\Tekkadan\CardioBreath_AI_for_medicine_hackathon\reports\cardiobreath_results.pdf
